<a href="https://colab.research.google.com/github/sarah-keren/ER-UMD-2018/blob/master/Running_the_day_ahead_market_cycle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*italicized text*

In [ ]:
!git clone https://github.com/CLAIR-LAB-TECHNION/energy-net.git

%cd energy-net
!pip install -e .

fatal: destination path 'energy-net' already exists and is not an empty directory.
/content/energy-net
Obtaining file:///content/energy-net
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 847.8/847.8 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.3/77.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.8/92.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━

# load players

In [ ]:
#TODO Itay
PCSs = ...  # load_pcss([config_files])  # including decision_horizon

In [ ]:
#TODO Matan
ISO = ... # ISOAgent(...)

# Training

In [ ]:
#TODO Matan X 3
import numpy as np

TIME_TICKS = 48

converge = False
data_24h = ...
cost_fn = ...
while not converge:
  buy_price, sell_price, power_gen = ISO.step(data_24h)
  loads = np.zeros(TIME_TICKS)
  for PCS in PCSs: # Itay
    loads += PCS.get_load(data_24h, buy_price, sell_price, horizon=TIME_TICKS)
  cost = cost_fn(buy_price, sell_price, power_gen, loads)
  ISO.train_step(cost)
  converge = ...  # check if converged

# Eval / Visualizations

In [ ]:
#TODO graph Itay
# cost bar chart ,demand X t  X 10


In [ ]:
#TODO graph - dispatch Matan
# power gen policy

In [ ]:
#TODO graph Sarah
# predicted demand

In [ ]:
#TODO graph throughout training Sarah
# cost

In [ ]:
#TODO eval

In [ ]:
import os
import yaml
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gymnasium as gym
# import the ISOs
from energy_net.dynamics.iso.hourly_pricing_iso import HourlyPricingISO
from energy_net.dynamics.iso.random_pricing_iso import RandomPricingISO
from energy_net.dynamics.iso.quadratic_pricing_iso import QuadraticPricingISO
from energy_net.dynamics.iso.time_of_use_pricing_iso import TimeOfUsePricingISO
from energy_net.dynamics.iso.dynamic_pricing_iso import DynamicPricingISO

# Reward import
from energy_net.rewards.cost_reward import CostReward
# Instantiate the CostReward class
reward_calc = CostReward()

sns.set_theme(style="whitegrid", font_scale=1.2)
%matplotlib inline

# ISOs

### HourlyPricingISO

The `HourlyPricingISO` sets a specific price for each hour of the day. We pass a dictionary `hourly_rates` that maps `hour -> price`.

Given a time (as a fraction of the day, where `0.0` is midnight and `1.0` is the next midnight), the ISO determines the current hour and sets a buy price. The sell price is a fraction (e.g., 90%) of the buy price.

The reward for a given transaction (`buy_amount`, `sell_amount`) is calculated as:

$$
\text{Reward} = (\text{buy_amount} \times \text{price_buy}) - (\text{sell_amount} \times \text{price_sell})
$$


In [ ]:
# HourlyPricingISO
# Here we define hourly rates that vary slightly by hour using a sine function for illustrative purposes.
hourly_rates = {hour: 50.0 + 10*np.sin(np.pi * hour/24) for hour in range(24)}

hp_iso = HourlyPricingISO(hourly_rates=hourly_rates)

times = np.linspace(0,1,100)  # Times as fractions of the day (0 = 0:00, 1 = 24:00)

hourly_rewards = []
hourly_prices = []
for t in times:
    pricing_func = hp_iso.get_pricing_function({'time': t})
    # To extract the current hour:
    current_hour = int(t*24) % 24
    price = hourly_rates.get(current_hour, 50.0)
    # Use the CostReward to compute reward
    info = {
        'pricing_function': pricing_func
    }
    reward = reward_calc.compute_reward(info)
    hourly_rewards.append(reward)
    hourly_prices.append(price)

hourly_prices = np.array(hourly_prices)
hours = times * 24

# Plotting the prices
plt.figure(figsize=(12,6))
plt.plot(hours, hourly_prices[:], label=' Price', color='blue')

plt.title('HourlyPricingISO Prices Over Time')
plt.xlabel('Hour of Day')
plt.ylabel('Price ($/MWh)')
plt.legend()
plt.grid(True)
plt.show()



## RandomPricingISO Plot

For the `RandomPricingISO`, the prices are random each time get_pricing_function is called. We'll just sample multiple pricing functions and record the resulting rewards distribution for a fixed buy/sell scenario.

In [ ]:
rp_iso = RandomPricingISO(min_price=40.0, max_price=60.0)

num_samples = 1000
buy_amount = 10

random_rewards = []
random_buy_prices = []
random_sell_prices = []

for _ in range(num_samples):
    pricing_func = rp_iso.get_pricing_function({})
    # We don't directly know the buy/sell price without re-generating them,
    # but we can assume it's stable for that call. Let's just call the function with 1 unit to guess prices.
    # Actually, to extract prices, let's trick the pricing function:
    # If we define a small system of equations:
    # reward = buy*price_buy - sell*price_sell
    # If we call pricing(1,0) = price_buy
    # pricing(0,1) = -price_sell
    price_buy = pricing_func(1)


    # Use the CostReward to compute reward
    info = {
        'pricing_function': pricing_func
    }
    reward = reward_calc.compute_reward(info)
    random_rewards.append(reward)
    random_buy_prices.append(price_buy)


plt.figure(figsize=(12,6))
sns.histplot(random_buy_prices, kde=True, color='blue', label='Prices')

plt.title('RandomPricingISO Price Distribution')
plt.xlabel('Price ($/MWh)')
plt.ylabel('Frequency')
plt.legend()
plt.show()



### QuadraticPricingISO

The `QuadraticPricingISO` computes the buy price using a quadratic function of demand:

$$
\text{price_buy}(d) = a \cdot d^2 + b \cdot d + c
$$

The sell price is a fixed fraction (e.g., 85%) of the buy price.

The reward calculation follows the same form as before:

$$
\text{Reward} = (\text{buy_amount} \times \text{price_buy}) - (\text{sell_amount} \times \text{price_sell})
$$


In [ ]:
demands = np.linspace(0,3,100)
param_sets = [
    (1.0, 0.0, 50.0),
    (0.5, 0.0, 50.0),
    (1.0, 2.0, 50.0),
    (1.0, 0.0, 60.0)
]

plt.figure(figsize=(12,6))
for (a, b, c) in param_sets:
    qp_iso = QuadraticPricingISO(a=a, b=b, c=c)
    q_prices_buy = []
    for d in demands:
        pricing_func = qp_iso.get_pricing_function({'demand': d})
        # Compute price_buy to plot (not directly needed for reward since we have pricing_func)
        price_buy = a*(d**2) + b*d + c
        q_prices_buy.append(price_buy)
    plt.plot(demands, q_prices_buy, label=f'a={a}, b={b}, c={c}')

plt.title('QuadraticPricingISO - Price under Different (a,b,c)')
plt.xlabel('Demand')
plt.ylabel('Price ($/MWh)')
plt.legend()
plt.grid(True)
plt.show()



## TimeOfUsePricingISO Plot

- The `TimeOfUsePricingISO` defines certain hours as peak and others as off-peak, with corresponding prices. Hours not in either category use a default price (average of peak and off-peak).
- We will show multiple instances of `TimeOfUsePricingISO` with different parameters to illustrate how these choices affect pricing.

In [ ]:
times = np.linspace(0,1,100)
hours = times*24
buy_amount = 10


# Define different TOU setups to compare
tou_setups = [
    {
        'peak_hours': [17,18,19],
        'off_peak_hours': [0,1,2,3,4,5],
        'peak_price': 60.0,
        'off_peak_price': 30.0,
        'label': 'Setup 1 (peak: 17-19, off-peak: 0-5)'
    },
    {
        'peak_hours': [16,17,18],
        'off_peak_hours': [22,23,0,1],
        'peak_price': 70.0,
        'off_peak_price': 20.0,
        'label': 'Setup 2 (peak:16-18, off-peak:22-1)'
    }
]

plt.figure(figsize=(12,6))
for setup in tou_setups:
    tou_iso = TimeOfUsePricingISO(
        peak_hours=setup['peak_hours'],
        off_peak_hours=setup['off_peak_hours'],
        peak_price=setup['peak_price'],
        off_peak_price=setup['off_peak_price']
    )
    tou_prices_buy = []
    for t in times:
        pricing_func = tou_iso.get_pricing_function({'time': t})
        # Determine buy price for plotting (as done before)
        current_hour = int(t*24)%24
        if current_hour in setup['peak_hours']:
            price_buy = setup['peak_price']
        elif current_hour in setup['off_peak_hours']:
            price_buy = setup['off_peak_price']
        else:
            price_buy = (setup['peak_price'] + setup['off_peak_price'])/2
        tou_prices_buy.append(price_buy)

    plt.plot(hours, tou_prices_buy, label=setup['label'])

plt.title('TimeOfUsePricingISO - Price under Different Setups')
plt.xlabel('Hour of Day')
plt.ylabel('Price ($/MWh)')
plt.legend()
plt.grid(True)
plt.show()

### DynamicPricingISO

The `DynamicPricingISO` adjusts prices based on both demand and supply, using an elasticity factor.

The price is computed as:

$$
\text{price_buy} = \text{base_price} \times \left[ 1 + \text{elasticity} \times (\text{demand} - \text{supply}) \right]
$$

The sell price is a fraction of the buy price.

By creating a 2D grid of `(demand, supply)` values and computing the resulting reward, we can visualize this relationship as a heatmap. This tool allows us to see how changes in these two factors affect the system's output (reward).

A heatmap is useful because it provides a quick overview of the reward landscape for different states, showing where high rewards or low rewards occur as we vary both demand and supply simultaneously.


In [ ]:
dp_iso = DynamicPricingISO(base_price=50.0, elasticity=0.5)

demands = np.linspace(0,2,50)
supplies = np.linspace(0,2,50)

buy_amount = 1


rewards_matrix = np.zeros((len(demands), len(supplies)))

for i, d in enumerate(demands):
    for j, s in enumerate(supplies):
        pricing_func = dp_iso.get_pricing_function({'demand': d, 'supply': s})
        info = {
            'pricing_function': pricing_func,
            'net_exchange': buy_amount
        }
        reward = reward_calc.compute_reward(info)
        rewards_matrix[i,j] = reward

plt.figure(figsize=(10,8))

ax = sns.heatmap(
    rewards_matrix,
    cmap='viridis',
    cbar_kws={"shrink": 0.8}
)

plt.title('DynamicPricingISO Reward Heatmap (Demand vs. Supply)')
plt.xlabel('Supply')
plt.ylabel('Demand')


num_ticks = 5

# Indices at which to place ticks
x_ticks = np.linspace(0, len(supplies)-1, num_ticks).astype(int)
y_ticks = np.linspace(0, len(demands)-1, num_ticks).astype(int)

# Corresponding values at these indices
x_tick_labels = [f"{supplies[i]:.1f}" for i in x_ticks]
y_tick_labels = [f"{demands[i]:.1f}" for i in y_ticks]

# Set the ticks
plt.xticks(x_ticks + 0.5, x_tick_labels, rotation=0)  # +0.5 to center labels between cells
plt.yticks(y_ticks + 0.5, y_tick_labels, rotation=0)

plt.show()


In [ ]:
import os
import pandas as pd

from stable_baselines3 import TD3
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy
import energy_net.env

# Make plots inline and set a nice style
%matplotlib inline
sns.set_theme(style="whitegrid")

# Training

In [ ]:
class RewardCallback(BaseCallback):
        """
        Custom callback for recording episode rewards during training.
        """
        def __init__(self, verbose=0):
            super(RewardCallback, self).__init__(verbose)
            self.rewards = []

        def _on_step(self) -> bool:
            # Check if an episode has finished
            for info in self.locals.get('infos', []):
                if 'episode' in info.keys():
                    self.rewards.append(info['episode']['r'])
            return True

reward_callback = RewardCallback()

In [ ]:
# Paths
env_id = 'PCSUnitEnv-v0'
env_config_path = 'configs/environment_config.yaml'
iso_config_path = 'configs/iso_config.yaml'
pcs_unit_config_path = 'configs/pcs_unit_config.yaml'

log_dir = 'logs/TD3_energy_net_env'
model_path = 'models/TD3_energy_net_env/TD3_energy_net_env'
os.makedirs(log_dir, exist_ok=True)
os.makedirs(os.path.dirname(model_path), exist_ok=True)

########################################
# Training
########################################

# Create training environment
train_env = gym.make(
    env_id,
    disable_env_checker=True,
    env_config_path=env_config_path,
    iso_config_path=iso_config_path,
    pcs_unit_config_path=pcs_unit_config_path,
    log_file=os.path.join(log_dir, 'train_environment.log')
)
train_env.reset(seed=42)
train_env.action_space.seed(42)
train_env.observation_space.seed(42)

# Instantiate TD3 model
model = TD3(
    'MlpPolicy',
    train_env,
    verbose=1,
    tensorboard_log=log_dir,
    seed=42
)

total_timesteps = 30_000  # Adjust as needed
print("Starting training...")
model.learn(total_timesteps=total_timesteps, callback=reward_callback, progress_bar=True)
print("Training completed.")

# Save the trained model
model.save(model_path)
print(f"Model saved at {model_path}.zip")

train_env.close()

In [ ]:
model = TD3.load(model_path)
eval_episodes = 1

eval_env = gym.make(
    env_id,
    disable_env_checker=True,
    env_config_path=env_config_path,
    iso_config_path=iso_config_path,
    pcs_unit_config_path=pcs_unit_config_path,
    log_file=os.path.join(log_dir, 'eval_environment.log')
)

mean_reward, std_reward = evaluate_policy(
        model,
        eval_env,
        n_eval_episodes=eval_episodes,
        deterministic=True
    )

print(f"Mean Reward: {mean_reward} +/- {std_reward}")

In [ ]:
if reward_callback.rewards:
        all_rewards = np.array(reward_callback.rewards)

        mean_reward = np.mean(all_rewards)
        std_reward = np.std(all_rewards) + 1e-8  # add small epsilon to avoid division by zero
        normalized_rewards = (all_rewards - mean_reward) / std_reward

        plt.figure(figsize=(12, 6))
        plt.plot(normalized_rewards, label='Episode Normalized rewards ')
        plt.xlabel('Episode')
        plt.ylabel('Reward')
        plt.title('Training Normalized rewards over Episodes')
        plt.legend()
        plt.grid(True)
        plt.savefig(os.path.join(log_dir, 'training_rewards.png'))
        plt.show()
else:
  print("No training rewards recorded.")

In [ ]:

obs, info = eval_env.reset()
done = False
truncated = False

socs = []
times = []
prices = []
self_productions = []
self_consumptions = []

while not (done or truncated):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = eval_env.step(action)
    # Observation format: [SOC, time_fraction, self_production, self_consumption]
    soc = obs[0]
    time_fraction = obs[1]
    self_production = obs[2]
    self_consumption = obs[3]

    hour_of_day = time_fraction * 24.0

    # Compute price using the iso object
    pricing_func = info.get('pricing_function')
    buy_price = pricing_func(1)  # buy price

    socs.append(soc)
    prices.append(buy_price)
    times.append(hour_of_day)
    self_productions.append(self_production)
    self_consumptions.append(self_consumption)

eval_env.close()

########################################
# Limit Data to 48 hours
########################################
mask = np.array(times) <= 48
times = np.array(times)[mask]
socs = np.array(socs)[mask]
prices = np.array(prices)[mask]
self_productions = np.array(self_productions)[mask]
self_consumptions = np.array(self_consumptions)[mask]

########################################
# Plotting
########################################

fig, ax1 = plt.subplots(figsize=(10,6))
color_sp = 'tab:gray'
color_sc = 'tab:cyan'

ax1.set_xlabel('Hour of Day')
ax1.set_ylabel('Self-Production (MWh)', color=color_sp)
ax1.plot(times, self_productions, color=color_sp, label='Self-Production')
ax1.tick_params(axis='y', labelcolor=color_sp)
ax1.grid(True)

ax2 = ax1.twinx()
ax2.set_ylabel('Self-Consumption (MWh)', color=color_sc)
ax2.plot(times, self_consumptions, color=color_sc, label='Self-Consumption')
ax2.tick_params(axis='y', labelcolor=color_sc)

fig.tight_layout()
plt.title('Self-Production and Self-Consumption vs. Time of Day')
plt.show()

